### Imports and device

In [1]:
import torch
import matplotlib.pyplot as plt

from data_preprocessing import create_dataloader, denormalize
from models import Generator, PatchGANDiscriminator

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: NVIDIA GeForce RTX 4090


### Dataset paths

In [2]:
from pathlib import Path

DATASET_ROOT = Path(
    r"C:\Users\samruddhi\Downloads\DATA266-Lab1-Pair45\task3_GAN\dataset"
)

# Correct domain assignment
MONET_SOURCE = DATASET_ROOT / "monet_jpg"
PHOTO_SOURCE = DATASET_ROOT / "photo_jpg"

SPLIT_ROOT = DATASET_ROOT / "split_data"

print("Monet source:", MONET_SOURCE)
print("Photo source:", PHOTO_SOURCE)

Monet source: C:\Users\samruddhi\Downloads\DATA266-Lab1-Pair45\task3_GAN\dataset\monet_jpg
Photo source: C:\Users\samruddhi\Downloads\DATA266-Lab1-Pair45\task3_GAN\dataset\photo_jpg


### Split the data

In [3]:
# import random
# import shutil


# def split_images(
#     source_dir,
#     domain_name,
#     output_root,
#     train_ratio=0.8,
#     validation_ratio=0.1,
#     seed=42
# ):
#     image_extensions = {
#         ".jpg",
#         ".jpeg",
#         ".png",
#         ".bmp",
#         ".webp"
#     }

#     image_paths = sorted([
#         path for path in source_dir.rglob("*")
#         if path.is_file()
#         and path.suffix.lower() in image_extensions
#     ])

#     if len(image_paths) == 0:
#         raise ValueError(
#             f"No images found in: {source_dir}"
#         )

#     rng = random.Random(seed)
#     rng.shuffle(image_paths)

#     total_images = len(image_paths)

#     train_end = int(total_images * train_ratio)

#     validation_end = int(
#         total_images
#         * (train_ratio + validation_ratio)
#     )

#     split_paths = {
#         "train": image_paths[:train_end],

#         "validation": image_paths[
#             train_end:validation_end
#         ],

#         "test": image_paths[
#             validation_end:
#         ]
#     }

#     for split_name, paths in split_paths.items():

#         destination_dir = (
#             output_root
#             / split_name
#             / domain_name
#         )

#         destination_dir.mkdir(
#             parents=True,
#             exist_ok=True
#         )

#         for image_path in paths:
#             shutil.copy2(
#                 image_path,
#                 destination_dir / image_path.name
#             )

#         print(
#             f"{domain_name} | "
#             f"{split_name}: {len(paths)} images"
#         )


# # Split photos independently
# split_images(
#     source_dir=PHOTO_SOURCE,
#     domain_name="photos",
#     output_root=SPLIT_ROOT,
#     seed=42
# )

# # Split Monet images independently
# split_images(
#     source_dir=MONET_SOURCE,
#     domain_name="monet",
#     output_root=SPLIT_ROOT,
#     seed=43
# )

photos | train: 5630 images
photos | validation: 704 images
photos | test: 704 images
monet | train: 240 images
monet | validation: 30 images
monet | test: 30 images


### Create the DataLoader

In [3]:
BATCH_SIZE = 10

train_loader = create_dataloader(
    domain_a_dir=SPLIT_ROOT / "train" / "photos",
    domain_b_dir=SPLIT_ROOT / "train" / "monet",
    batch_size=BATCH_SIZE,
    train=True,
    shuffle=True,
    num_workers=0
)

print("Batch size:", BATCH_SIZE)
print("Number of training batches:", len(train_loader))

Batch size: 10
Number of training batches: 563


## Architecture testing

### Load one batch

In [5]:
batch = next(iter(train_loader))

real_A = batch["A"].to(device)  # Photos
real_B = batch["B"].to(device)  # Monet paintings

print("Photo batch shape:", real_A.shape)
print("Monet batch shape:", real_B.shape)

Photo batch shape: torch.Size([10, 3, 256, 256])
Monet batch shape: torch.Size([10, 3, 256, 256])


### Create the four CycleGAN networks

In [4]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

G_AB = Generator(
    input_channels=3,
    output_channels=3,
    num_residual_blocks=9
).to(device)

G_BA = Generator(
    input_channels=3,
    output_channels=3,
    num_residual_blocks=9
).to(device)

D_A = PatchGANDiscriminator(
    input_channels=3
).to(device)

D_B = PatchGANDiscriminator(
    input_channels=3
).to(device)

print("Models created successfully.")
print("Device:", device)

Models created successfully.
Device: cuda


### Define losses

In [5]:
import torch.nn as nn

# Adversarial loss
# It encourages generated images to look realistic to the discriminator.
criterion_gan = nn.MSELoss()

# Cycle-consistency loss
# This checks whether the image can be translated and then returned close to its original form
criterion_cycle = nn.L1Loss()

# Identity loss
# This checks whether a generator preserves an image when it already receives an image from its target domain
criterion_identity = nn.L1Loss()

# Loss weights
lambda_cycle = 10.0
lambda_identity = 5.0

### Optimizers

In [6]:
import itertools
import torch.optim as optim

learning_rate = 0.0002
beta1 = 0.5
beta2 = 0.999

# One optimizer for both generators
optimizer_G = optim.Adam(
    itertools.chain(
        G_AB.parameters(),
        G_BA.parameters()
    ),
    lr=learning_rate,
    betas=(beta1, beta2)
)

# Separate optimizer for each discriminator
optimizer_D_A = optim.Adam(
    D_A.parameters(),
    lr=learning_rate,
    betas=(beta1, beta2)
)

optimizer_D_B = optim.Adam(
    D_B.parameters(),
    lr=learning_rate,
    betas=(beta1, beta2)
)

- optimizer_G   → updates both generators
- optimizer_D_A → updates the photo discriminator
- optimizer_D_B → updates the Monet discriminator

### Testing on one sample

In [7]:
batch = next(iter(train_loader))

real_A = batch["A"].to(device)
real_B = batch["B"].to(device)

print("Photo batch shape:", real_A.shape)
print("Monet batch shape:", real_B.shape)
print("Photo value range:", real_A.min().item(), real_A.max().item())
print("Monet value range:", real_B.min().item(), real_B.max().item())

with torch.no_grad():
    fake_B = G_AB(real_A)
    fake_A = G_BA(real_B)

print("Generated Monet shape:", fake_B.shape)
print("Generated Photo shape:", fake_A.shape)
print("Generated value range:", fake_B.min().item(), fake_B.max().item())

Photo batch shape: torch.Size([10, 3, 256, 256])
Monet batch shape: torch.Size([10, 3, 256, 256])
Photo value range: -1.0 1.0
Monet value range: -1.0 1.0
Generated Monet shape: torch.Size([10, 3, 256, 256])
Generated Photo shape: torch.Size([10, 3, 256, 256])
Generated value range: -0.9997686147689819 0.9980267882347107


### Train the Model

In [8]:
import os
import time
import torch


# ============================================================
# Training settings
# ============================================================

num_epochs = 100
patience = 10

BATCH_SIZE = 10

checkpoint_dir = "checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

print("Checkpoint folder:", os.path.abspath(checkpoint_dir))


# ============================================================
# Clear unused GPU memory
# ============================================================

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ============================================================
# Helper functions
# ============================================================

def set_requires_grad(models, requires_grad):
    for model in models:
        for parameter in model.parameters():
            parameter.requires_grad = requires_grad


def safe_torch_save(checkpoint, filename):
    """
    Save to a temporary file first.
    Replace the final file only after saving succeeds.
    """
    temporary_filename = filename + ".tmp"

    torch.save(
        checkpoint,
        temporary_filename
    )

    os.replace(
        temporary_filename,
        filename
    )


# ============================================================
# Loss history
# ============================================================

history = {
    "generator": [],
    "discriminator_A": [],
    "discriminator_B": [],
    "cycle": [],
    "identity": []
}


# ============================================================
# Early stopping variables
# ============================================================

best_monitor_score = float("inf")
epochs_without_improvement = 0


# ============================================================
# Start training
# ============================================================

training_start_time = time.time()

print("Training started.", flush=True)
print(f"Total epochs: {num_epochs}", flush=True)
print(f"Batch size: {BATCH_SIZE}", flush=True)
print(f"Batches per epoch: {len(train_loader)}", flush=True)
print(f"Device: {device}", flush=True)
print("-" * 70, flush=True)


for epoch in range(num_epochs):

    epoch_start_time = time.time()

    total_loss_G = 0.0
    total_loss_D_A = 0.0
    total_loss_D_B = 0.0
    total_loss_cycle = 0.0
    total_loss_identity = 0.0

    number_of_batches = 0

    print(
        f"Starting Epoch [{epoch + 1}/{num_epochs}]...",
        flush=True
    )


    # ========================================================
    # Process every training batch
    # ========================================================

    for batch in train_loader:

        real_A = batch["A"].to(device)
        real_B = batch["B"].to(device)


        # ====================================================
        # Step 1: Train both generators
        # ====================================================

        set_requires_grad([D_A, D_B], False)

        optimizer_G.zero_grad(set_to_none=True)

        # Photo → Monet
        fake_B = G_AB(real_A)

        # Monet → Photo
        fake_A = G_BA(real_B)

        # Photo → Monet → Photo
        reconstructed_A = G_BA(fake_B)

        # Monet → Photo → Monet
        reconstructed_B = G_AB(fake_A)

        # Identity mappings
        identity_A = G_BA(real_A)
        identity_B = G_AB(real_B)

        # Discriminator predictions for generated images
        prediction_fake_B = D_B(fake_B)
        prediction_fake_A = D_A(fake_A)

        # Generator adversarial losses
        loss_G_AB = criterion_gan(
            prediction_fake_B,
            torch.ones_like(prediction_fake_B)
        )

        loss_G_BA = criterion_gan(
            prediction_fake_A,
            torch.ones_like(prediction_fake_A)
        )

        # Cycle-consistency losses
        loss_cycle_A = criterion_cycle(
            reconstructed_A,
            real_A
        )

        loss_cycle_B = criterion_cycle(
            reconstructed_B,
            real_B
        )

        loss_cycle_total = (
            loss_cycle_A + loss_cycle_B
        )

        # Identity losses
        loss_identity_A = criterion_identity(
            identity_A,
            real_A
        )

        loss_identity_B = criterion_identity(
            identity_B,
            real_B
        )

        loss_identity_total = (
            loss_identity_A + loss_identity_B
        )

        # Total generator loss
        loss_G = (
            loss_G_AB
            + loss_G_BA
            + lambda_cycle * loss_cycle_total
            + lambda_identity * loss_identity_total
        )

        # Update generators
        loss_G.backward()
        optimizer_G.step()


        # ====================================================
        # Step 2: Train photo discriminator D_A
        # ====================================================

        set_requires_grad([D_A, D_B], True)

        optimizer_D_A.zero_grad(set_to_none=True)

        prediction_real_A = D_A(real_A)
        prediction_fake_A = D_A(fake_A.detach())

        loss_D_A_real = criterion_gan(
            prediction_real_A,
            torch.ones_like(prediction_real_A)
        )

        loss_D_A_fake = criterion_gan(
            prediction_fake_A,
            torch.zeros_like(prediction_fake_A)
        )

        loss_D_A = 0.5 * (
            loss_D_A_real + loss_D_A_fake
        )

        loss_D_A.backward()
        optimizer_D_A.step()


        # ====================================================
        # Step 3: Train Monet discriminator D_B
        # ====================================================

        optimizer_D_B.zero_grad(set_to_none=True)

        prediction_real_B = D_B(real_B)
        prediction_fake_B = D_B(fake_B.detach())

        loss_D_B_real = criterion_gan(
            prediction_real_B,
            torch.ones_like(prediction_real_B)
        )

        loss_D_B_fake = criterion_gan(
            prediction_fake_B,
            torch.zeros_like(prediction_fake_B)
        )

        loss_D_B = 0.5 * (
            loss_D_B_real + loss_D_B_fake
        )

        loss_D_B.backward()
        optimizer_D_B.step()


        # ====================================================
        # Store losses
        # ====================================================

        total_loss_G += loss_G.item()
        total_loss_D_A += loss_D_A.item()
        total_loss_D_B += loss_D_B.item()
        total_loss_cycle += loss_cycle_total.item()
        total_loss_identity += loss_identity_total.item()

        number_of_batches += 1


        # ====================================================
        # Print batch losses
        # ====================================================

        print(
            f"Epoch [{epoch + 1}/{num_epochs}] | "
            f"Batch [{number_of_batches}/{len(train_loader)}] | "
            f"Generator: {loss_G.item():.4f} | "
            f"D_A: {loss_D_A.item():.4f} | "
            f"D_B: {loss_D_B.item():.4f} | "
            f"Cycle: {loss_cycle_total.item():.4f} | "
            f"Identity: {loss_identity_total.item():.4f}",
            flush=True
        )


        # Release large tensor references
        del (
            real_A,
            real_B,
            fake_A,
            fake_B,
            reconstructed_A,
            reconstructed_B,
            identity_A,
            identity_B,
            prediction_fake_A,
            prediction_fake_B,
            prediction_real_A,
            prediction_real_B
        )


    # ========================================================
    # Calculate average epoch losses
    # ========================================================

    if number_of_batches == 0:
        print("No batches were processed. Training stopped.")
        break

    average_loss_G = (
        total_loss_G / number_of_batches
    )

    average_loss_D_A = (
        total_loss_D_A / number_of_batches
    )

    average_loss_D_B = (
        total_loss_D_B / number_of_batches
    )

    average_loss_cycle = (
        total_loss_cycle / number_of_batches
    )

    average_loss_identity = (
        total_loss_identity / number_of_batches
    )


    # Store history
    history["generator"].append(average_loss_G)
    history["discriminator_A"].append(average_loss_D_A)
    history["discriminator_B"].append(average_loss_D_B)
    history["cycle"].append(average_loss_cycle)
    history["identity"].append(average_loss_identity)


    # ========================================================
    # Print epoch summary
    # ========================================================

    epoch_time = time.time() - epoch_start_time

    print("\nEpoch summary:", flush=True)

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] | "
        f"Time: {epoch_time:.2f} seconds",
        flush=True
    )

    print(
        f"Average Generator loss: "
        f"{average_loss_G:.6f}",
        flush=True
    )

    print(
        f"Average Photo discriminator loss: "
        f"{average_loss_D_A:.6f}",
        flush=True
    )

    print(
        f"Average Monet discriminator loss: "
        f"{average_loss_D_B:.6f}",
        flush=True
    )

    print(
        f"Average Cycle-consistency loss: "
        f"{average_loss_cycle:.6f}",
        flush=True
    )

    print(
        f"Average Identity loss: "
        f"{average_loss_identity:.6f}",
        flush=True
    )


    # ========================================================
    # Save latest model checkpoint
    # ========================================================

    # Model-only checkpoint to save disk space
    latest_checkpoint = {
        "epoch": epoch + 1,
        "G_AB": G_AB.state_dict(),
        "G_BA": G_BA.state_dict(),
        "D_A": D_A.state_dict(),
        "D_B": D_B.state_dict(),
        "history": history
    }

    latest_path = os.path.join(
        checkpoint_dir,
        "latest_cyclegan.pth"
    )

    safe_torch_save(
        latest_checkpoint,
        latest_path
    )


    # ========================================================
    # Early stopping monitor
    # ========================================================

    monitor_score = (
        average_loss_cycle
        + average_loss_identity
    )


    # ========================================================
    # Save best model
    # ========================================================

    if monitor_score < best_monitor_score:

        best_monitor_score = monitor_score
        epochs_without_improvement = 0

        # Model-only checkpoint to save disk space
        best_checkpoint = {
            "epoch": epoch + 1,
            "G_AB": G_AB.state_dict(),
            "G_BA": G_BA.state_dict(),
            "D_A": D_A.state_dict(),
            "D_B": D_B.state_dict(),
            "history": history,
            "best_monitor_score": best_monitor_score
        }

        best_path = os.path.join(
            checkpoint_dir,
            "best_cyclegan.pth"
        )

        safe_torch_save(
            best_checkpoint,
            best_path
        )

        print(
            f"Best model saved. "
            f"Monitor score: {best_monitor_score:.6f}",
            flush=True
        )

    else:

        epochs_without_improvement += 1

        print(
            f"No improvement for "
            f"{epochs_without_improvement} epoch(s).",
            flush=True
        )

        if epochs_without_improvement >= patience:

            print(
                "Early stopping triggered.",
                flush=True
            )

            break


    print("-" * 70, flush=True)


# ============================================================
# Training completed
# ============================================================

total_training_time = (
    time.time() - training_start_time
)

print(
    f"Training completed in "
    f"{total_training_time / 60:.2f} minutes.",
    flush=True
)

print(
    f"Latest checkpoint saved at: {latest_path}",
    flush=True
)

if "best_path" in locals():
    print(
        f"Best checkpoint saved at: {best_path}",
        flush=True
    )

Checkpoint folder: C:\Users\samruddhi\Downloads\DATA266-Lab1-Pair45\task3_GAN\Samruddhi\src\checkpoints
Training started.
Total epochs: 100
Batch size: 10
Batches per epoch: 563
Device: cuda
----------------------------------------------------------------------
Starting Epoch [1/100]...
Epoch [1/100] | Batch [1/563] | Generator: 18.1930 | D_A: 0.9301 | D_B: 0.5590 | Cycle: 1.0401 | Identity: 1.0443
Epoch [1/100] | Batch [2/563] | Generator: 16.2459 | D_A: 3.5508 | D_B: 3.9280 | Cycle: 0.8508 | Identity: 0.8331
Epoch [1/100] | Batch [3/563] | Generator: 15.9579 | D_A: 1.4652 | D_B: 0.6793 | Cycle: 0.9487 | Identity: 0.9505
Epoch [1/100] | Batch [4/563] | Generator: 15.1303 | D_A: 0.7172 | D_B: 0.4804 | Cycle: 0.9179 | Identity: 0.9531
Epoch [1/100] | Batch [5/563] | Generator: 14.5823 | D_A: 0.4635 | D_B: 0.9951 | Cycle: 0.8491 | Identity: 0.8882
Epoch [1/100] | Batch [6/563] | Generator: 14.1428 | D_A: 0.4603 | D_B: 0.7386 | Cycle: 0.8214 | Identity: 0.8490
Epoch [1/100] | Batch [7/563

RuntimeError: [enforce fail at inline_container.cc:672] . unexpected pos 64 vs 0

### Training the model to save states

In [11]:
# import os
# import time
# import torch


# # ============================================================
# # Training settings
# # ============================================================

# num_epochs = 100
# patience = 10

# checkpoint_dir = "checkpoints"
# os.makedirs(checkpoint_dir, exist_ok=True)


# # ============================================================
# # Reduce unnecessary memory usage
# # ============================================================

# if torch.cuda.is_available():
#     torch.cuda.empty_cache()


# # ============================================================
# # Helper function
# # ============================================================

# def set_requires_grad(models, requires_grad):
#     for model in models:
#         for parameter in model.parameters():
#             parameter.requires_grad = requires_grad


# # ============================================================
# # Loss history
# # ============================================================

# history = {
#     "generator": [],
#     "discriminator_A": [],
#     "discriminator_B": [],
#     "cycle": [],
#     "identity": []
# }


# # ============================================================
# # Early stopping variables
# # ============================================================

# best_monitor_score = float("inf")
# epochs_without_improvement = 0


# # ============================================================
# # Start training
# # ============================================================

# training_start_time = time.time()

# print("Training started.", flush=True)
# print(f"Total epochs: {num_epochs}", flush=True)
# print(f"Batches per epoch: {len(train_loader)}", flush=True)
# print(f"Device: {device}", flush=True)
# print("-" * 70, flush=True)


# for epoch in range(num_epochs):

#     epoch_start_time = time.time()

#     total_loss_G = 0.0
#     total_loss_D_A = 0.0
#     total_loss_D_B = 0.0
#     total_loss_cycle = 0.0
#     total_loss_identity = 0.0

#     number_of_batches = 0

#     print(
#         f"Starting Epoch [{epoch + 1}/{num_epochs}]...",
#         flush=True
#     )


#     # ========================================================
#     # Process all batches
#     # ========================================================

#     for batch in train_loader:

#         real_A = batch["A"].to(device, non_blocking=True)
#         real_B = batch["B"].to(device, non_blocking=True)


#         # ====================================================
#         # Step 1: Train the generators
#         # ====================================================

#         set_requires_grad([D_A, D_B], False)

#         optimizer_G.zero_grad(set_to_none=True)

#         # Photo → Monet
#         fake_B = G_AB(real_A)

#         # Monet → Photo
#         fake_A = G_BA(real_B)

#         # Photo → Monet → Photo
#         reconstructed_A = G_BA(fake_B)

#         # Monet → Photo → Monet
#         reconstructed_B = G_AB(fake_A)

#         # Identity mappings
#         identity_A = G_BA(real_A)
#         identity_B = G_AB(real_B)

#         # Discriminator predictions for generated images
#         prediction_fake_B = D_B(fake_B)
#         prediction_fake_A = D_A(fake_A)

#         # Adversarial generator losses
#         loss_G_AB = criterion_gan(
#             prediction_fake_B,
#             torch.ones_like(prediction_fake_B)
#         )

#         loss_G_BA = criterion_gan(
#             prediction_fake_A,
#             torch.ones_like(prediction_fake_A)
#         )

#         # Cycle-consistency losses
#         loss_cycle_A = criterion_cycle(
#             reconstructed_A,
#             real_A
#         )

#         loss_cycle_B = criterion_cycle(
#             reconstructed_B,
#             real_B
#         )

#         loss_cycle_total = loss_cycle_A + loss_cycle_B

#         # Identity losses
#         loss_identity_A = criterion_identity(
#             identity_A,
#             real_A
#         )

#         loss_identity_B = criterion_identity(
#             identity_B,
#             real_B
#         )

#         loss_identity_total = loss_identity_A + loss_identity_B

#         # Total generator loss
#         loss_G = (
#             loss_G_AB
#             + loss_G_BA
#             + lambda_cycle * loss_cycle_total
#             + lambda_identity * loss_identity_total
#         )

#         loss_G.backward()
#         optimizer_G.step()


#         # ====================================================
#         # Step 2: Train discriminator D_A
#         # ====================================================

#         set_requires_grad([D_A, D_B], True)

#         optimizer_D_A.zero_grad(set_to_none=True)

#         prediction_real_A = D_A(real_A)
#         prediction_fake_A = D_A(fake_A.detach())

#         loss_D_A_real = criterion_gan(
#             prediction_real_A,
#             torch.ones_like(prediction_real_A)
#         )

#         loss_D_A_fake = criterion_gan(
#             prediction_fake_A,
#             torch.zeros_like(prediction_fake_A)
#         )

#         loss_D_A = 0.5 * (
#             loss_D_A_real + loss_D_A_fake
#         )

#         loss_D_A.backward()
#         optimizer_D_A.step()


#         # ====================================================
#         # Step 3: Train discriminator D_B
#         # ====================================================

#         optimizer_D_B.zero_grad(set_to_none=True)

#         prediction_real_B = D_B(real_B)
#         prediction_fake_B = D_B(fake_B.detach())

#         loss_D_B_real = criterion_gan(
#             prediction_real_B,
#             torch.ones_like(prediction_real_B)
#         )

#         loss_D_B_fake = criterion_gan(
#             prediction_fake_B,
#             torch.zeros_like(prediction_fake_B)
#         )

#         loss_D_B = 0.5 * (
#             loss_D_B_real + loss_D_B_fake
#         )

#         loss_D_B.backward()
#         optimizer_D_B.step()


#         # ====================================================
#         # Store batch losses
#         # ====================================================

#         total_loss_G += loss_G.item()
#         total_loss_D_A += loss_D_A.item()
#         total_loss_D_B += loss_D_B.item()
#         total_loss_cycle += loss_cycle_total.item()
#         total_loss_identity += loss_identity_total.item()

#         number_of_batches += 1


#         # ====================================================
#         # Print batch progress
#         # ====================================================

#         print(
#             f"Epoch [{epoch + 1}/{num_epochs}] | "
#             f"Batch [{number_of_batches}/{len(train_loader)}] | "
#             f"Generator: {loss_G.item():.4f} | "
#             f"D_A: {loss_D_A.item():.4f} | "
#             f"D_B: {loss_D_B.item():.4f} | "
#             f"Cycle: {loss_cycle_total.item():.4f} | "
#             f"Identity: {loss_identity_total.item():.4f}",
#             flush=True
#         )


#         # Release references to large tensors
#         del (
#             real_A,
#             real_B,
#             fake_A,
#             fake_B,
#             reconstructed_A,
#             reconstructed_B,
#             identity_A,
#             identity_B,
#             prediction_fake_A,
#             prediction_fake_B,
#             prediction_real_A,
#             prediction_real_B
#         )


#     # ========================================================
#     # Calculate average epoch losses
#     # ========================================================

#     average_loss_G = total_loss_G / number_of_batches
#     average_loss_D_A = total_loss_D_A / number_of_batches
#     average_loss_D_B = total_loss_D_B / number_of_batches
#     average_loss_cycle = total_loss_cycle / number_of_batches
#     average_loss_identity = total_loss_identity / number_of_batches

#     history["generator"].append(average_loss_G)
#     history["discriminator_A"].append(average_loss_D_A)
#     history["discriminator_B"].append(average_loss_D_B)
#     history["cycle"].append(average_loss_cycle)
#     history["identity"].append(average_loss_identity)


#     # ========================================================
#     # Print epoch summary
#     # ========================================================

#     epoch_time = time.time() - epoch_start_time

#     print("\nEpoch summary:", flush=True)
#     print(
#         f"Epoch [{epoch + 1}/{num_epochs}] | "
#         f"Time: {epoch_time:.2f} seconds",
#         flush=True
#     )
#     print(
#         f"Average Generator loss: {average_loss_G:.6f}",
#         flush=True
#     )
#     print(
#         f"Average Photo discriminator loss: {average_loss_D_A:.6f}",
#         flush=True
#     )
#     print(
#         f"Average Monet discriminator loss: {average_loss_D_B:.6f}",
#         flush=True
#     )
#     print(
#         f"Average Cycle-consistency loss: {average_loss_cycle:.6f}",
#         flush=True
#     )
#     print(
#         f"Average Identity loss: {average_loss_identity:.6f}",
#         flush=True
#     )


#     # ========================================================
#     # Save latest checkpoint
#     # ========================================================

#     latest_checkpoint = {
#         "epoch": epoch + 1,
#         "G_AB": G_AB.state_dict(),
#         "G_BA": G_BA.state_dict(),
#         "D_A": D_A.state_dict(),
#         "D_B": D_B.state_dict(),
#         "optimizer_G": optimizer_G.state_dict(),
#         "optimizer_D_A": optimizer_D_A.state_dict(),
#         "optimizer_D_B": optimizer_D_B.state_dict(),
#         "history": history
#     }

#     torch.save(
#         latest_checkpoint,
#         os.path.join(
#             checkpoint_dir,
#             "latest_cyclegan.pth"
#         )
#     )


#     # ========================================================
#     # Early stopping
#     # ========================================================

#     monitor_score = (
#         average_loss_cycle
#         + average_loss_identity
#     )

#     if monitor_score < best_monitor_score:

#         best_monitor_score = monitor_score
#         epochs_without_improvement = 0

#         best_checkpoint = {
#             "epoch": epoch + 1,
#             "G_AB": G_AB.state_dict(),
#             "G_BA": G_BA.state_dict(),
#             "D_A": D_A.state_dict(),
#             "D_B": D_B.state_dict(),
#             "optimizer_G": optimizer_G.state_dict(),
#             "optimizer_D_A": optimizer_D_A.state_dict(),
#             "optimizer_D_B": optimizer_D_B.state_dict(),
#             "history": history,
#             "best_monitor_score": best_monitor_score
#         }

#         torch.save(
#             best_checkpoint,
#             os.path.join(
#                 checkpoint_dir,
#                 "best_cyclegan.pth"
#             )
#         )

#         print(
#             f"Best model saved. "
#             f"Monitor score: {best_monitor_score:.6f}",
#             flush=True
#         )

#     else:

#         epochs_without_improvement += 1

#         print(
#             f"No improvement for "
#             f"{epochs_without_improvement} epoch(s).",
#             flush=True
#         )

#         if epochs_without_improvement >= patience:

#             print(
#                 "Early stopping triggered.",
#                 flush=True
#             )

#             break

#     print("-" * 70, flush=True)


# # ============================================================
# # Training completed
# # ============================================================

# total_training_time = time.time() - training_start_time

# print(
#     f"Training completed in "
#     f"{total_training_time / 60:.2f} minutes.",
#     flush=True
# )

# print(
#     f"Latest checkpoint saved at: "
#     f"{checkpoint_dir}/latest_cyclegan.pth",
#     flush=True
# )

# print(
#     f"Best checkpoint saved at: "
#     f"{checkpoint_dir}/best_cyclegan.pth",
#     flush=True
# )

Training started.
Total epochs: 100
Batches per epoch: 563
Device: cuda
----------------------------------------------------------------------
Starting Epoch [1/100]...
Epoch [1/100] | Batch [1/563] | Generator: 20.6408 | D_A: 0.6259 | D_B: 0.6097 | Cycle: 1.2333 | Identity: 1.2319
Epoch [1/100] | Batch [2/563] | Generator: 16.6384 | D_A: 4.6112 | D_B: 1.4713 | Cycle: 0.9199 | Identity: 0.9171
Epoch [1/100] | Batch [3/563] | Generator: 14.1998 | D_A: 0.9431 | D_B: 1.4957 | Cycle: 0.8771 | Identity: 0.8863
Epoch [1/100] | Batch [4/563] | Generator: 14.9088 | D_A: 0.4381 | D_B: 0.7480 | Cycle: 0.9313 | Identity: 0.9134
Epoch [1/100] | Batch [5/563] | Generator: 14.7513 | D_A: 0.5770 | D_B: 0.7112 | Cycle: 0.8794 | Identity: 0.8748
Epoch [1/100] | Batch [6/563] | Generator: 13.6616 | D_A: 0.3428 | D_B: 0.7417 | Cycle: 0.8182 | Identity: 0.8036
Epoch [1/100] | Batch [7/563] | Generator: 13.2769 | D_A: 0.4414 | D_B: 0.4562 | Cycle: 0.8214 | Identity: 0.8031
Epoch [1/100] | Batch [8/563] | G

RuntimeError: [enforce fail at inline_container.cc:672] . unexpected pos 135130688 vs 135130580